In [ ]:
import os
from pathlib import Path

DATA_ROOT = Path(os.environ.get('B_CELL_ATLAS_DATA', 'data'))

## install packages if needed
!pip install scanpy anndata numpy pandas scvi mudata matplotlib seaborn

In [ ]:
## import all packages
import scanpy as sc
import scvi
import mudata as md
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import re
import os, glob

In [ ]:
### settings
warnings.filterwarnings("ignore")
sc.settings.verbosity = 2
sc.settings.figdir = str(DATA_ROOT / "GSE295491/figures/01_qc/")
sc.settings.set_figure_params(dpi=120, frameon=False, figsize=(6, 4))
Path(sc.settings.figdir).mkdir(parents=True, exist_ok=True)
DATA_DIR = str(DATA_ROOT / 'GSE295491')

In [ ]:
## creating data dir
!mkdir str(DATA_ROOT)

In [ ]:
## downloading the data
!wget -P str(DATA_ROOT) https://ftp.ncbi.nlm.nih.gov/geo/series/GSE295nnn/GSE295491/suppl/GSE295491_RAW.tar

In [ ]:
# The series matrix file has all sample titles and metadata, parse through tbi
!cat {DATA_DIR}/GSE295491_series_matrix.txt | grep -E "!Sample_title|!Sample_geo_accession" | head -5

In [ ]:
test = sc.read_10x_mtx(
    DATA_DIR,
    var_names='gene_symbols',
    prefix='GSM8951228_Pool119-3_',
    cache=False
)
print(test)

In [ ]:
# Parse directly from what the series matrix gave us
titles = ["P15 HC-MBL RNA","P15 CLL RNA","P14 HC-MBL RNA","P14 CLL RNA",
          "P8 HC-MBL RNA","P10 HC-MBL RNA","P7 HC-MBL RNA","P9 HC-MBL RNA",
          "P23 HC-MBL RNA","P23 CLL RNA","P20 HC-MBL RNA","P20 CLL RNA",
          "P16 HC-MBL RNA","P16 CLL RNA","P12 HC-MBL RNA","P24 CLL RNA",
          "P19 HC-MBL RNA","P19 CLL RNA","P21 HC-MBL RNA","P21 CLL RNA",
          "P13 HC-MBL RNA","P11 HC-MBL RNA","P22 HC-MBL RNA","P22 CLL RNA",
          "P6 LC-MBL RNA","P9 HC-MBL RNA replicate","P2 LC-MBL RNA",
          "P1 LC-MBL RNA","P4 LC-MBL RNA","P3 LC-MBL RNA","P5 LC-MBL RNA",
          "P19 HC-MBL BCR","P19 CLL BCR","P21 HC-MBL BCR","P21 CLL BCR",
          "P13 HC-MBL BCR","P11 HC-MBL BCR","P22 HC-MBL BCR","P22 CLL BCR",
          "P15 HC-MBL BCR","P15 CLL BCR","P14 HC-MBL BCR","P14 CLL BCR",
          "P8 HC-MBL BCR","P10 HC-MBL BCR","P7 HC-MBL BCR","P9 HC-MBL BCR",
          "P23 HC-MBL BCR","P23 CLL BCR","P20 HC-MBL BCR","P20 CLL BCR",
          "P16 HC-MBL BCR","P16 CLL BCR","P12 HC-MBL BCR","P24 CLL BCR",
          "P6 LC-MBL BCR","P9 HC-MBL BCR replicate","P2 LC-MBL BCR",
          "P1 LC-MBL BCR","P4 LC-MBL BCR","P3 LC-MBL BCR","P5 LC-MBL BCR"]

gsms = ["GSM8951228","GSM8951229","GSM8951230","GSM8951231","GSM8951232",
        "GSM8951233","GSM8951234","GSM8951235","GSM8951236","GSM8951237",
        "GSM8951238","GSM8951239","GSM8951240","GSM8951241","GSM8951242",
        "GSM8951243","GSM8951244","GSM8951245","GSM8951246","GSM8951247",
        "GSM8951248","GSM8951249","GSM8951250","GSM8951251","GSM8951252",
        "GSM8951253","GSM8951254","GSM8951255","GSM8951256","GSM8951257",
        "GSM8951258","GSM8951259","GSM8951260","GSM8951261","GSM8951262",
        "GSM8951263","GSM8951264","GSM8951265","GSM8951266","GSM8951267",
        "GSM8951268","GSM8951269","GSM8951270","GSM8951271","GSM8951272",
        "GSM8951273","GSM8951274","GSM8951275","GSM8951276","GSM8951277",
        "GSM8951278","GSM8951279","GSM8951280","GSM8951281","GSM8951282",
        "GSM8951283","GSM8951284","GSM8951285","GSM8951286","GSM8951287",
        "GSM8951288","GSM8951289"]

meta = pd.DataFrame({'gsm': gsms, 'title': titles})
meta['patient_id']     = meta['title'].str.extract(r'(P\d+)')
meta['disease_stage']  = meta['title'].str.extract(r'(HC-MBL|LC-MBL|CLL|Normal)')
meta['data_type']      = meta['title'].str.extract(r'(RNA|BCR)')
meta['is_replicate']   = meta['title'].str.contains('replicate')

print(meta['disease_stage'].value_counts())
print("\nFull metadata:")
print(meta.to_string(index=False))

In [ ]:

DATA_DIR  = str(DATA_ROOT / 'GSE295491')
H5AD_DIR  = str(DATA_ROOT / 'GSE295491/h5ad_per_sample')
os.makedirs(H5AD_DIR, exist_ok=True)

rna_meta = meta[meta['is_replicate'] == False].copy()


def mad_filter(x, n_mads=5):
    med = np.median(x)
    mad = np.median(np.abs(x - med))
    return med - n_mads * mad,  med + n_mads * mad

for _, row in rna_meta.iterrows():
    out_path = f"{H5AD_DIR}/{row['gsm']}.h5ad"
    if os.path.exists(out_path):
        print(f"  skip {row['gsm']} (already done)")
        continue

    mtx_files = glob.glob(f"{DATA_DIR}/{row['gsm']}_*_matrix.mtx.gz")
    if not mtx_files:
        print(f"  MISSING: {row['gsm']}")
        continue

    prefix = mtx_files[0].split('/')[-1].split('_matrix')[0] + '_'
    adata  = sc.read_10x_mtx(DATA_DIR, var_names='gene_symbols',
                              prefix=prefix, cache=False)

    ## adding metdata
    adata.obs['gsm']           = row['gsm']
    adata.obs['patient_id']    = row['patient_id']
    adata.obs['disease_stage'] = row['disease_stage']
    adata.obs_names = [f"{row['gsm']}_{bc}" for bc in adata.obs_names]

    ## compute QC metrics
    adata.var['mt'] = adata.var_names.str.startswith('MT-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'],
                               percent_top=None, log1p=False, inplace=True)
    n_before = adata.n_obs
    lo_c, hi_c = mad_filter(adata.obs['total_counts'],      n_mads=5)
    lo_g, hi_g = mad_filter(adata.obs['n_genes_by_counts'], n_mads=5)
    _,    hi_m = mad_filter(adata.obs['pct_counts_mt'],     n_mads=3)

    adata = adata[
        (adata.obs['total_counts']      >= max(lo_c, 500))  &
        (adata.obs['total_counts']      <= hi_c)            &
        (adata.obs['n_genes_by_counts'] >= max(lo_g, 200))  &
        (adata.obs['n_genes_by_counts'] <= hi_g)            &
        (adata.obs['pct_counts_mt']     <= min(hi_m, 20))
    ].copy()

    print(f"  ✓ {row['title']}: {n_before} → {adata.n_obs} cells after QC")

    adata.write_h5ad(out_path)
    del adata

In [ ]:
## should be 30 samples
!ls str(DATA_ROOT)/h5ad_per_sample/ | wc -l

In [ ]:

## chekc if there re any empty files befor concat
H5AD_DIR = str(DATA_ROOT / 'GSE295491/h5ad_per_sample')
files = sorted(glob.glob(f"{H5AD_DIR}/*.h5ad"))
print("Found", len(files), "files\n")

bad = []
for f in files:
    try:
        a = ad.read_h5ad(f, backed='r')
        print(f"{f}: {a.n_obs} cells × {a.n_vars} genes")
        if a.n_obs == 0 or a.n_vars == 0:
            bad.append(f)
    except Exception as e:
        print(f"{f}: ERROR → {e}")
        bad.append(f)

print("\nBad files:", bad)

In [ ]:

CLEAN_DIR = str(DATA_ROOT / 'GSE295491/h5ad_per_sample_clean')
OUT_PATH  = str(DATA_ROOT / 'GSE295491/combined_qc_raw.h5ad')

files = sorted(glob.glob(f"{CLEAN_DIR}/*.h5ad"))
print("Found", len(files), "clean files")

adatas = []
keys   = []

for f in files:
    a = ad.read_h5ad(f)  
    
    a = ad.AnnData(X=a.X, obs=a.obs.copy(), var=a.var.copy())
    adatas.append(a)
    keys.append(os.path.basename(f).replace('.h5ad', ''))
    print(f"{os.path.basename(f)}: {a.n_obs} cells × {a.n_vars} genes")

adata = ad.concat(
    adatas,
    axis=0,              
    join='outer',       
    merge='same',        
    label='gsm',         
    keys=keys,
    index_unique=None,   
)

print("Combined:", adata.n_obs, "cells ×", adata.n_vars, "genes")
adata.write_h5ad(OUT_PATH)
print("Saved:", OUT_PATH)